**Replicate Built-in Losses**
- **Task:** Generate dummy predictions and targets. Implement Mean Squared Error (MSE) and Binary Cross-Entropy (BCE) using pure NumPy. Compare your outputs to torch.nn.functional.mse_loss and torch.nn.functional.binary_cross_entropy to ensure they match exactly.
- **Why:** Loss functions are just math formulas; this demystifies framework implementations.

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# ==========================================
# 1. Generate Dummy Predictions and Targets
# ==========================================

# Regression data (for MSE)
# Shapes: (N, D) where N is batch size and D is output dimensions
y_pred_reg = np.random.randn(5, 3)
y_true_reg = np.random.randn(5, 3)

# Classification data (for BCE)
# Probabilities bounded between 0 and 1, targets are binary (0 or 1)
y_pred_clf = np.random.uniform(0.01, 0.99, size=(5, 1))
y_true_clf = np.random.choice([0.0, 1.0], size=(5, 1))


# ==========================================
# 2. Implement Loss Functions in NumPy
# ==========================================

def numpy_mse_loss(input, target, reduction='mean'):
    """Calculates Mean Squared Error."""
    loss = (input - target) ** 2
    
    if reduction == 'mean':
        return np.mean(loss)
    elif reduction == 'sum':
        return np.sum(loss)
    else:
        return loss

def numpy_bce_loss(input, target, reduction='mean'):
    """Calculates Binary Cross-Entropy."""
    # Formula: - (target * log(input) + (1 - target) * log(1 - input))
    loss = - (target * np.log(input) + (1.0 - target) * np.log(1.0 - input))
    
    if reduction == 'mean':
        return np.mean(loss)
    elif reduction == 'sum':
        return np.sum(loss)
    else:
        return loss


# ==========================================
# 3. Compare and Verify with PyTorch
# ==========================================

# --- MSE Verification ---
np_mse = numpy_mse_loss(y_pred_reg, y_true_reg, reduction='mean')

torch_mse = F.mse_loss(
    torch.tensor(y_pred_reg), 
    torch.tensor(y_true_reg), 
    reduction='mean'
).item()

# --- BCE Verification ---
np_bce = numpy_bce_loss(y_pred_clf, y_true_clf, reduction='mean')

torch_bce = F.binary_cross_entropy(
    torch.tensor(y_pred_clf), 
    torch.tensor(y_true_clf), 
    reduction='mean'
).item()


# ==========================================
# 4. Print Results
# ==========================================

print("--- Mean Squared Error (MSE) ---")
print(f"NumPy Implementation:   {np_mse:.6f}")
print(f"PyTorch Implementation: {torch_mse:.6f}")
print(f"Matches Exactly:        {np.isclose(np_mse, torch_mse)}")

print("\n--- Binary Cross-Entropy (BCE) ---")
print(f"NumPy Implementation:   {np_bce:.6f}")
print(f"PyTorch Implementation: {torch_bce:.6f}")
print(f"Matches Exactly:        {np.isclose(np_bce, torch_bce)}")

--- Mean Squared Error (MSE) ---
NumPy Implementation:   1.508081
PyTorch Implementation: 1.508081
Matches Exactly:        True

--- Binary Cross-Entropy (BCE) ---
NumPy Implementation:   1.368689
PyTorch Implementation: 1.368689
Matches Exactly:        True


**The LogSumExp Trick (Numerical Stability)**
- **Task:** Implement Categorical Cross-Entropy loss with a Softmax activation in NumPy. Then, pass in a prediction vector with very large numbers (e.g., [1000, 2000, 3000]). Observe the NaN output due to floating-point overflow. Research and implement the "LogSumExp trick" to fix this numerical instability.
- **Why:** Numerical stability is the hardest silent bug in deep learning. This teaches you how frameworks handle extremes.

In [3]:
import numpy as np
import torch
import torch.nn.functional as F

# ==========================================
# 1. Generate Extreme Dummy Predictions & Targets
# ==========================================

y_pred_extreme = np.array([[1000.0, 2000.0, 3000.0]])
y_true_index = np.array([2]) 

num_classes = y_pred_extreme.shape[1]
y_true_one_hot = np.eye(num_classes)[y_true_index]


# ==========================================
# 2. Truly Naive Implementation (Left raw to show the original float overflow)
# ==========================================

def naive_categorical_crossentropy(logits, targets):
    # Pure raw math without any stability adjustments
    exps = np.exp(logits)  # <-- Overflows to inf here
    softmax_out = exps / np.sum(exps, axis=1, keepdims=True)  # <-- inf / inf = NaN
    
    loss = -np.sum(targets * np.log(softmax_out), axis=1)
    return np.mean(loss)


# ==========================================
# 3. Stable Implementation (LogSumExp Trick)
# ==========================================

def stable_categorical_crossentropy(logits, targets):
    # 1. Shift logits to prevent overflow
    max_logits = np.max(logits, axis=1, keepdims=True)
    shifted_logits = logits - max_logits
    
    # 2. Combined Log-Softmax computation to prevent underflow to 0.0
    log_sum_exp = np.log(np.sum(np.exp(shifted_logits), axis=1, keepdims=True))
    log_softmax = shifted_logits - log_sum_exp  # Direct subtraction avoids log(0)
    
    # 3. Cross entropy computation
    loss = -np.sum(targets * log_softmax, axis=1)
    return np.mean(loss)


# ==========================================
# 4. Compare Outputs and Verify with PyTorch
# ==========================================

# Run implementations
stable_loss = stable_categorical_crossentropy(y_pred_extreme, y_true_one_hot)

torch_loss = F.cross_entropy(
    torch.tensor(y_pred_extreme),
    torch.tensor(y_true_index, dtype=torch.long)
).item()
# ==========================================
# 5. Print Results
# ==========================================

print("--- Categorical Cross-Entropy Stability Test ---")
print(f"Naive Implementation Output:  {naive_loss}")
print(f"Stable NumPy (LogSumExp):     {stable_loss:.6f}")
print(f"PyTorch Implementation:       {torch_loss:.6f}")
print(f"Stable Matches PyTorch:       {np.isclose(stable_loss, torch_loss)}")

--- Categorical Cross-Entropy Stability Test ---
Naive Implementation Output:  nan
Stable NumPy (LogSumExp):     0.000000
PyTorch Implementation:       0.000000
Stable Matches PyTorch:       True
